# Exoplanet Discovery Biases

## Statistical comparison of NASA Exoplanet Archive discovery methods

This notebook studies how observed exoplanet properties differ across discovery methods in the NASA Exoplanet Archive. The goal is not to infer the full intrinsic population of planets, but to quantify how the observed catalog changes when planets are grouped by discovery technique.

### Scientific questions
1. Do observed orbital period, radius, and mass distributions depend on discovery method?
2. Does a simple log-log mass-radius relation provide a useful global summary, or do method-specific subsets behave differently?
3. Can a lightweight Monte Carlo selection model reproduce the qualitative direction of the observed biases?


## 0. Background: exoplanet discovery methods

Exoplanets are found with techniques that are sensitive to different parts of parameter space.

- **Transit** searches detect periodic dips in stellar brightness and are naturally biased toward short-period planets with relatively large radii.
- **Radial velocity** searches measure the stellar motion along the line of sight and are more sensitive to massive planets on closer orbits.
- **Microlensing** can reveal planets at larger separations, although catalog entries often have fewer complete physical parameters.
- **Direct imaging, timing, astrometry, and other methods** remain scientifically important, but their sample sizes or data completeness can be limiting for this comparison.

These selection effects mean that catalog distributions of period, radius, and mass should be interpreted as observed distributions, not as an unbiased census of all exoplanets.


## 1. Libraries and settings


In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from iminuit import Minuit
from iminuit.cost import LeastSquares

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 100)


## 2. Download data from the NASA Exoplanet Archive

The notebook queries the `pscomppars` table through the archive TAP endpoint and keeps the columns needed for the analysis.


In [ ]:
QUERY = """
select
    pl_name,
    discoverymethod,
    disc_year,
    pl_orbper,
    pl_rade,
    pl_bmasse,
    pl_orbeccen,
    st_mass,
    st_rad,
    sy_dist
from pscomppars
where discoverymethod is not null
"""

URL = (
    "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    "?query=" + requests.utils.quote(QUERY) +
    "&format=csv"
)

def download_exoplanet_data(url: str = URL) -> pd.DataFrame:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    return pd.read_csv(io.StringIO(response.text))


In [ ]:
df_raw = download_exoplanet_data()
df_raw.head()


In [ ]:
print("Shape:", df_raw.shape)
df_raw.info()


### Discovery methods in the raw data

The table below counts planets by discovery method in the downloaded catalog. The main analysis keeps methods with enough planets to support meaningful statistical comparisons.


In [ ]:
method_counts_raw = (
    df_raw["discoverymethod"]
    .value_counts()
    .rename_axis("discoverymethod")
    .reset_index(name="n_planets")
)
method_counts_raw


In [ ]:
MIN_METHOD_COUNT = 100

method_selection = method_counts_raw.copy()
method_selection["included_by_count"] = method_selection["n_planets"] >= MIN_METHOD_COUNT
method_selection["selection_note"] = np.where(
    method_selection["included_by_count"],
    "included: enough planets for the main analysis",
    "excluded from the main analysis: too few planets",
)
method_selection


## 3. Initial data cleaning


In [ ]:
NUMERIC_COLUMNS = [
    "disc_year",
    "pl_orbper",
    "pl_rade",
    "pl_bmasse",
    "pl_orbeccen",
    "st_mass",
    "st_rad",
    "sy_dist",
]

def prepare_dataset(df: pd.DataFrame) -> pd.DataFrame:
    clean = df.copy()

    for col in NUMERIC_COLUMNS:
        if col in clean.columns:
            clean[col] = pd.to_numeric(clean[col], errors="coerce")

    clean = clean.drop_duplicates(subset=["pl_name"])
    clean["discoverymethod"] = clean["discoverymethod"].astype(str).str.strip()

    for col in ["pl_orbper", "pl_rade", "pl_bmasse", "st_mass", "st_rad", "sy_dist"]:
        if col in clean.columns:
            clean.loc[clean[col] <= 0, col] = np.nan

    return clean


In [ ]:
df = prepare_dataset(df_raw)
print("Shape after initial cleaning:", df.shape)
df.head()


In [ ]:
missing_summary = (
    df[["pl_orbper", "pl_rade", "pl_bmasse", "pl_orbeccen", "st_mass", "st_rad", "sy_dist"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
)
missing_summary


### Data completeness by discovery method

Before running statistical tests, it is useful to inspect how many non-null values remain for each physical variable and discovery method. This prevents over-interpreting comparisons based on very small subsets.


In [ ]:
missing_by_method = (
    df.groupby("discoverymethod")[["pl_orbper", "pl_rade", "pl_bmasse", "pl_orbeccen", "st_mass", "st_rad", "sy_dist"]]
    .apply(lambda g: g.notna().mean())
    .sort_index()
)
missing_by_method


In [ ]:
nonmissing_counts = (
    df.groupby("discoverymethod")[["pl_orbper", "pl_rade", "pl_bmasse", "pl_orbeccen", "st_mass", "st_rad", "sy_dist"]]
    .count()
    .sort_values("pl_orbper", ascending=False)
)
nonmissing_counts


## 4. Main sample selection


In [ ]:
def filter_main_methods(df: pd.DataFrame, min_count: int = 100) -> pd.DataFrame:
    counts = df["discoverymethod"].value_counts()
    valid_methods = counts[counts >= min_count].index
    return df[df["discoverymethod"].isin(valid_methods)].copy()

def add_log_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for source, target in [
        ("pl_orbper", "log_orbper"),
        ("pl_rade", "log_rade"),
        ("pl_bmasse", "log_bmasse"),
    ]:
        if source in out.columns:
            out[target] = np.log10(out[source])

    return out


In [ ]:
df = filter_main_methods(df, min_count=MIN_METHOD_COUNT)
df = add_log_columns(df)

print("Selected sample shape:", df.shape)
df["discoverymethod"].value_counts()


## 5. Descriptive analysis


In [ ]:
def summarize_by_method(df: pd.DataFrame) -> pd.DataFrame:
    variables = ["log_orbper", "log_rade", "log_bmasse"]
    rows = []

    for method, subdf in df.groupby("discoverymethod"):
        row = {"discoverymethod": method, "n_planets": len(subdf)}
        for col in variables:
            data = subdf[col].dropna()
            row[f"{col}_n"] = len(data)
            row[f"{col}_mean"] = data.mean()
            row[f"{col}_std"] = data.std(ddof=1)
            row[f"{col}_median"] = data.median()
            row[f"{col}_q25"] = data.quantile(0.25)
            row[f"{col}_q75"] = data.quantile(0.75)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("n_planets", ascending=False)

summary_table = summarize_by_method(df)
summary_table


In [ ]:
def plot_histograms_by_method(df: pd.DataFrame, column: str, bins: int = 30) -> None:
    plt.figure(figsize=(8, 5))

    for method, subdf in df.groupby("discoverymethod"):
        data = subdf[column].dropna()
        if len(data) > 0:
            plt.hist(data, bins=bins, density=True, histtype="step", label=method)

    plt.xlabel(column)
    plt.ylabel("density")
    plt.title(f"Distribution of {column} by discovery method")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_histograms_by_method(df, "log_orbper")
plot_histograms_by_method(df, "log_rade")
plot_histograms_by_method(df, "log_bmasse")


In [ ]:
def plot_boxplot_by_method(df: pd.DataFrame, column: str) -> None:
    groups = [subdf[column].dropna().values for _, subdf in df.groupby("discoverymethod")]
    labels = [method for method, _ in df.groupby("discoverymethod")]

    plt.figure(figsize=(8, 5))
    plt.boxplot(groups, tick_labels=labels, showfliers=False)
    plt.ylabel(column)
    plt.title(f"Robust summaries of {column} by method")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

plot_boxplot_by_method(df, "log_orbper")
plot_boxplot_by_method(df, "log_rade")
plot_boxplot_by_method(df, "log_bmasse")


In [ ]:
def plot_ecdf_by_method(df: pd.DataFrame, column: str) -> None:
    plt.figure(figsize=(8, 5))

    for method, subdf in df.groupby("discoverymethod"):
        x = np.sort(subdf[column].dropna().values)
        if len(x) == 0:
            continue
        y = np.arange(1, len(x) + 1) / len(x)
        plt.step(x, y, where="post", label=method)

    plt.xlabel(column)
    plt.ylabel("ECDF")
    plt.title(f"ECDF of {column} by discovery method")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_ecdf_by_method(df, "log_orbper")
plot_ecdf_by_method(df, "log_rade")
plot_ecdf_by_method(df, "log_bmasse")


### Relationships among quantitative variables

Correlation and covariance provide a compact summary of how orbital period, radius, and mass co-vary in the selected sample.


In [ ]:
log_variables = ["log_orbper", "log_rade", "log_bmasse"]

correlation_matrix = df[log_variables].corr()
covariance_matrix = df[log_variables].cov()

correlation_matrix


In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(correlation_matrix, vmin=-1, vmax=1)
plt.colorbar(label="correlation")
plt.xticks(range(len(log_variables)), log_variables, rotation=30)
plt.yticks(range(len(log_variables)), log_variables)
plt.title("Matrice di correlation")
plt.tight_layout()
plt.show()


## 6. Hypothesis tests across discovery methods


In [ ]:
def pairwise_ks_tests(df: pd.DataFrame, column: str, min_n: int = 30) -> pd.DataFrame:
    methods = sorted(df["discoverymethod"].dropna().unique())
    rows = []

    for i, m1 in enumerate(methods):
        for m2 in methods[i + 1:]:
            x1 = df.loc[df["discoverymethod"] == m1, column].dropna()
            x2 = df.loc[df["discoverymethod"] == m2, column].dropna()

            if len(x1) < min_n or len(x2) < min_n:
                continue

            stat, pvalue = stats.ks_2samp(x1, x2)
            rows.append({
                "column": column,
                "method_1": m1,
                "method_2": m2,
                "ks_stat": stat,
                "pvalue": pvalue,
                "n1": len(x1),
                "n2": len(x2),
            })

    return pd.DataFrame(rows).sort_values("pvalue")


In [ ]:
ks_orbper = pairwise_ks_tests(df, "log_orbper")
ks_rade = pairwise_ks_tests(df, "log_rade")
ks_mass = pairwise_ks_tests(df, "log_bmasse")

ks_orbper.head(10)


In [ ]:
ks_rade.head(10)


In [ ]:
ks_mass.head(10)


### Chi-square test on binned variables

The Kolmogorov-Smirnov test compares continuous distributions. As a complementary check, the notebook bins each variable into quantiles and tests whether discovery method and variable bin are independent.


In [ ]:
def chi_square_independence_binned(df: pd.DataFrame, column: str, q: int = 4):
    tmp = df.dropna(subset=["discoverymethod", column]).copy()
    tmp["bin"] = pd.qcut(tmp[column], q=q, duplicates="drop")

    observed = pd.crosstab(tmp["discoverymethod"], tmp["bin"])
    obs = observed.values.astype(float)
    expected = np.outer(obs.sum(axis=1), obs.sum(axis=0)) / obs.sum()

    chi2_obs = ((obs - expected) ** 2 / expected).sum()
    ndof = (obs.shape[0] - 1) * (obs.shape[1] - 1)
    pvalue = stats.chi2.sf(chi2_obs, ndof)

    result = pd.DataFrame({
        "column": [column],
        "chi2": [chi2_obs],
        "ndof": [ndof],
        "pvalue": [pvalue],
        "min_expected": [expected.min()],
    })
    return observed, result

period_table, chi_period = chi_square_independence_binned(df, "log_orbper", q=4)
chi_period


In [ ]:
radius_table, chi_radius = chi_square_independence_binned(df, "log_rade", q=4)
mass_table, chi_mass = chi_square_independence_binned(df, "log_bmasse", q=4)
pd.concat([chi_period, chi_radius, chi_mass], ignore_index=True)


## 7. Bootstrap estimates for summary statistics


In [ ]:
def bootstrap_statistic(values: np.ndarray, statistic=np.median, n_boot: int = 2000, seed: int = 42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError("Empty array after removing NaN values.")

    samples = np.empty(n_boot)
    for i in range(n_boot):
        boot = rng.choice(values, size=len(values), replace=True)
        samples[i] = statistic(boot)

    return samples

def bootstrap_ci(values: np.ndarray, statistic=np.median, n_boot: int = 2000, alpha: float = 0.05):
    samples = bootstrap_statistic(values, statistic=statistic, n_boot=n_boot)
    lower = np.quantile(samples, alpha / 2)
    upper = np.quantile(samples, 1 - alpha / 2)
    return lower, upper, samples


In [ ]:
example_method = df["discoverymethod"].value_counts().index[0]
example_values = df.loc[df["discoverymethod"] == example_method, "log_orbper"].dropna().values

ci_low, ci_high, boot_samples = bootstrap_ci(example_values, statistic=np.median)
print("Method:", example_method)
print("95% bootstrap CI for the median of log_orbper:", (ci_low, ci_high))


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(boot_samples, bins=40)
plt.axvline(ci_low, linestyle="--")
plt.axvline(ci_high, linestyle="--")
plt.title(f"Bootstrap distribution of the median log_orbper - {example_method}")
plt.xlabel("mediana bootstrap")
plt.ylabel("frequency")
plt.tight_layout()
plt.show()


## 8. Mass-radius relation


In [ ]:
def mass_radius_sample(df: pd.DataFrame) -> pd.DataFrame:
    sub = df.dropna(subset=["pl_bmasse", "pl_rade"]).copy()
    sub = sub[(sub["pl_bmasse"] > 0) & (sub["pl_rade"] > 0)]
    sub["log_mass"] = np.log10(sub["pl_bmasse"])
    sub["log_radius"] = np.log10(sub["pl_rade"])
    return sub

def linear_model(x, a, b):
    return a + b * x

def fit_mass_radius_linear(df: pd.DataFrame):
    x = df["log_mass"].values
    y = df["log_radius"].values

    if len(y) < 3:
        raise ValueError("At least three points are required to estimate intercept, slope, and residuals.")

    first_cost = LeastSquares(x, y, np.ones_like(y), linear_model)
    first_fit = Minuit(first_cost, a=0.0, b=0.3)
    first_fit.migrad()
    first_fit.hesse()

    first_residuals = y - linear_model(x, first_fit.values["a"], first_fit.values["b"])
    ndof = len(y) - 2
    residual_std = np.sqrt(np.sum(first_residuals ** 2) / ndof)
    if not np.isfinite(residual_std) or residual_std <= 0:
        residual_std = 1.0

    yerr = np.full_like(y, residual_std)
    cost = LeastSquares(x, y, yerr, linear_model)
    fit = Minuit(cost, a=first_fit.values["a"], b=first_fit.values["b"])
    fit.migrad()
    fit.hesse()

    a = float(fit.values["a"])
    b = float(fit.values["b"])
    y_pred = linear_model(x, a, b)
    residuals = y - y_pred

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    chi2_value = float(fit.fval)
    pvalue = stats.chi2.sf(chi2_value, ndof)

    covariance = np.array(fit.covariance) if fit.covariance is not None else np.full((2, 2), np.nan)

    return {
        "intercept": a,
        "slope": b,
        "intercept_err": float(fit.errors["a"]),
        "slope_err": float(fit.errors["b"]),
        "r2": float(r2),
        "chi2": chi2_value,
        "ndof": int(ndof),
        "chi2_ndof": chi2_value / ndof,
        "pvalue_chi2": float(pvalue),
        "residual_std": float(residual_std),
        "cov": covariance,
        "minuit": fit,
        "y_pred": y_pred,
    }

def plot_mass_radius_fit(df: pd.DataFrame, fit_result: dict, title: str = "Mass-radius fit with Minuit") -> None:
    plt.figure(figsize=(7, 5))
    plt.scatter(df["log_mass"], df["log_radius"], alpha=0.35, s=12)

    x_grid = np.linspace(df["log_mass"].min(), df["log_mass"].max(), 200)
    y_grid = linear_model(x_grid, fit_result["intercept"], fit_result["slope"])

    plt.plot(x_grid, y_grid, color="black")
    plt.xlabel("log10(Earth masses)")
    plt.ylabel("log10(Earth radii)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
mr = mass_radius_sample(df)
print("Mass-radius sample size:", mr.shape)

fit_global = fit_mass_radius_linear(mr)
fit_global


In [ ]:
plot_mass_radius_fit(mr, fit_global, title="Global mass-radius fit with Minuit")


In [ ]:
mr = mr.copy()
mr["fit_residual"] = mr["log_radius"] - linear_model(
    mr["log_mass"], fit_global["intercept"], fit_global["slope"]
)

plt.figure(figsize=(7, 5))
plt.scatter(mr["log_mass"], mr["fit_residual"], alpha=0.35, s=12)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("log10(Earth masses)")
plt.ylabel("residuo su log10(Earth radii)")
plt.title("Mass-radius fit residuals")
plt.tight_layout()
plt.show()


In [ ]:
def fit_by_method_mass_radius(df: pd.DataFrame, min_count: int = 30) -> pd.DataFrame:
    rows = []

    for method, subdf in df.groupby("discoverymethod"):
        if len(subdf) < min_count:
            continue
        result = fit_mass_radius_linear(subdf)
        rows.append({
            "method": method,
            "n": len(subdf),
            "intercept": result["intercept"],
            "slope": result["slope"],
            "r2": result["r2"],
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False)


In [ ]:
fit_methods = fit_by_method_mass_radius(mr, min_count=30)
fit_methods


### Bootstrap and Minuit

The main mass-radius fit uses `Minuit` with a least-squares cost. For the bootstrap distribution of the slope, `scipy.stats.linregress` is used as a faster estimator on many resampled datasets.


In [ ]:
def bootstrap_mass_radius_slope(df: pd.DataFrame, n_boot: int = 1000, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    slopes = np.empty(n_boot)

    arr = df[["log_mass", "log_radius"]].dropna().values
    n = len(arr)

    for i in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boot = arr[idx]
        slope, intercept, r_value, p_value, std_err = stats.linregress(boot[:, 0], boot[:, 1])
        slopes[i] = slope

    return slopes


In [ ]:
slope_samples = bootstrap_mass_radius_slope(mr, n_boot=1000)
np.quantile(slope_samples, [0.025, 0.5, 0.975])


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(slope_samples, bins=40)
plt.title("Bootstrap distribution of the mass-radius slope")
plt.xlabel("slope")
plt.ylabel("frequency")
plt.tight_layout()
plt.show()


## 9. Monte Carlo simulation of observational bias


In [ ]:
def simulate_planet_population(n: int = 20000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    log_mass = rng.normal(loc=0.5, scale=0.7, size=n)
    log_period = rng.normal(loc=1.5, scale=0.8, size=n)
    log_radius = 0.3 + 0.35 * log_mass + rng.normal(0, 0.15, size=n)

    return pd.DataFrame({
        "log_mass": log_mass,
        "log_period": log_period,
        "log_radius": log_radius,
    })

def transit_detection_probability(pop: pd.DataFrame) -> np.ndarray:
    score = 1.2 * pop["log_radius"] - 0.9 * pop["log_period"]
    prob = 1 / (1 + np.exp(-score))
    return np.clip(prob, 0, 1)

def rv_detection_probability(pop: pd.DataFrame) -> np.ndarray:
    score = 1.1 * pop["log_mass"] - 0.5 * pop["log_period"]
    prob = 1 / (1 + np.exp(-score))
    return np.clip(prob, 0, 1)

def observe_population(pop: pd.DataFrame, prob: np.ndarray, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    u = rng.random(len(pop))
    return pop.loc[u < prob].copy()


In [ ]:
pop = simulate_planet_population()
transit_obs = observe_population(pop, transit_detection_probability(pop), seed=1)
rv_obs = observe_population(pop, rv_detection_probability(pop), seed=2)

print("Total population:", len(pop))
print("Observed by transit:", len(transit_obs))
print("Observed by radial velocity:", len(rv_obs))


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(transit_obs["log_period"], bins=40, density=True, histtype="step", label="Transit")
plt.hist(rv_obs["log_period"], bins=40, density=True, histtype="step", label="Radial Velocity")
plt.xlabel("log_period")
plt.ylabel("density")
plt.title("Selection-bias effect on orbital period")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(transit_obs["log_radius"], bins=40, density=True, histtype="step", label="Transit")
plt.hist(rv_obs["log_radius"], bins=40, density=True, histtype="step", label="Radial Velocity")
plt.xlabel("log_radius")
plt.ylabel("density")
plt.title("Selection-bias effect on radius")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Conclusions

This analysis is designed to separate three ideas that are often mixed together in catalog studies:

- Discovery method is a major selection variable, so observed distributions of period, radius, and mass should not be read as direct intrinsic population distributions.
- Non-parametric tests and binned chi-square tests provide complementary evidence about method-dependent differences in the observed sample.
- A simple log-log mass-radius fit is useful as a descriptive baseline, but the residuals and method-specific fits help reveal where the global summary is too coarse.
- The Monte Carlo section is intentionally qualitative: it shows how different detection probabilities can reshape an underlying population into different observed samples.

Because the notebook downloads live archive data, exact counts and p-values can change as the NASA Exoplanet Archive is updated.
